In [304]:
# Configuration

project = "NevinTure/java-course-2024"
# zero-shot, few-shot, chain-of-thought, tree-of-thought
prompting_strategy = "zero-shot"
# openai/gpt-5.2, z-ai/glm-5, anthropic/claude-opus-4.5, openai/gpt-oss-120b, z-ai/glm-4.7-flash
model = "openai/gpt-5.2"
debug = True

In [305]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [306]:
ground_truth_df = pd.read_csv('../datasets/training-set.csv')
project_ground_truth_df = ground_truth_df[ground_truth_df["Project"] == project]
project_ground_truth_df = project_ground_truth_df.drop(columns=['Project', 'Comment'])
project_ground_truth_df.sort_values(by=["Antipattern", "File", "Line from"], ascending=True, ignore_index=True)

,Antipattern,File,Line from,Line to
0,31 Flavors,scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/TgChat.java,63,63
1,Beware of the Unknown,scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/StackoverflowQuestion.java,86,86
2,Beware of the Unknown,scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/TgChat.java,63,63
3,ID Required,scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/GitRepository.java,61,61
4,ID Required,scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/Link.java,60,60
5,ID Required,scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/StackoverflowQuestion.java,61,61
6,ID Required,scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/TgChat.java,58,58
7,Implicit Columns,scrapper/src/main/java/edu/java/scrapper/repositories/jooq/JooqChatLinkRepository.java,55,55
8,Implicit Columns,scrapper/src/main/java/edu/java/scrapper/repositories/jooq/JooqChatRepository.java,56,56
9,Implicit Columns,scrapper/src/main/java/edu/java/scrapper/repositories/jooq/JooqLinkRepository.java,32,32


In [307]:
import glob, os
project_dir = f"../repositories/{project.replace("/", "_")}"
all_java_file_paths = glob.glob(project_dir + "/**/*.java", recursive=True)
print("Total Java files: " + str(len(all_java_file_paths)))

Total Java files: 197


In [308]:
applicable_file_paths = []

# Excluding files generated based on system schemas and database extensions/migration tools
excluded_path_fragments = [
    # Java
    'module-info.java',
    'package-info.java',
    
    # MySQL
    'information_schema/InformationSchema.java',
    'information_schema/tables',
    'mysql/Mysql.java',
    'mysql/tables',
    'performance_schema/PerformanceSchema.java',
    'performance_schema/tables',
    'sys/Sys.java',
    'sys/tables',

    # Flyway
    'FlywaySchemaHistory.java',

    # Liquibase
    'Databasechangelog.java',
    'Databasechangeloglock.java',

    # PostgreSQL
    'information_schema/Domains.java',
    'pg_catalog/PgCatalog.java',
    'pg_catalog/tables',

    # Quartz
    'tables/Qrtz',

    # PGP
    'PgpArmorHeaders.java',

    # Dblink
    'tables/Dblink',

    # hstore
    'tables/Each.java',
    'tables/Skeys.java',
    'tables/Svals.java'
]

# Including files that contain a reference to jOOQ or a potential reference to a generated file (the latter may include false positives)
included_phrases = [
    'org.jooq',
    'Tables',
    'tables'
]

# Excluding generated classes, which contain redundant information that doesn't help with detection
excluded_phrases = [
    'extends TableRecordImpl',
    'extends UpdatableRecordImpl',
    'extends org.jooq.impl.UpdatableRecordImpl',
    'extends CatalogImpl',
    'extends SchemaImpl',
    'extends UDTImpl',
    'extends UDTRecordImpl',
    'extends DAOImpl',
    'extends AbstractRoutine',
    'implements EnumType',
    'implements DeserializableJooqEnum',
    'public class Indexes',
    'public class Keys',
    'public class Routines',
    'public class Sequences',
    'public class Tables',
    'public class Domains',
    'class AbstractSpringDAOImpl',
    'extends AbstractSpringDAOImpl',
    'tables.pojos;',
    'tables.interfaces;',
    'Targeted by JavaCPP'
]

for file_path in all_java_file_paths:
    if all(fragment not in file_path for fragment in excluded_path_fragments) and os.path.isfile(file_path):
        with open(file_path, encoding='latin-1') as file:
            file_contents = file.read()
            if any(phrase in file_contents for phrase in included_phrases) and all(phrase not in file_contents for phrase in excluded_phrases):
                applicable_file_paths.append(file_path)

print("Applicable Java files: " + str(len(applicable_file_paths)))

Applicable Java files: 15


In [309]:
applicable_file_paths

['repositories/NevinTure_java-course-2024/scrapper-jooq/src/main/java/edu/java/jooq/JooqCodegen.java',
 'repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/configuration/JooqConfig.java',
 'repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/GitRepository.java',
 'repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/Link.java',
 'repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/StackoverflowQuestion.java',
 'repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/TgChat.java',
 'repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/model/jooq/tables/TgChatLink.java',
 'repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/repositories/jooq/JooqChatLinkRepository.java',
 'repositories/NevinTure_java-course-2024/scrapper/s

In [310]:
keys_file_paths = []

for file_path in all_java_file_paths:
    if os.path.isfile(file_path):
        with open(file_path, encoding='latin-1') as file:
            file_contents = file.read()
            if 'public class Keys ' in file_contents:
                keys_file_paths.append(file_path)

print("Keys files:")
for file_path in keys_file_paths:
    print(file_path)

Keys files:
repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/model/jooq/Keys.java


In [311]:
def find_closest_keys_file(file_path):
    closest_keys_file_path = None
    closest_common_prefix_length = 0

    for keys_file_path in keys_file_paths:
        common_prefix_length = len(os.path.commonprefix([file_path, keys_file_path]))
        if common_prefix_length > closest_common_prefix_length:
            closest_keys_file_path = keys_file_path
            closest_common_prefix_length = common_prefix_length
        
    return closest_keys_file_path

In [312]:
ddl_prompt_path = f"../prompts/{prompting_strategy}/ddl.md"
dml_dql_prompt_path = f"../prompts/{prompting_strategy}/dml-dql.md"
ddl_prompt_template = None
dml_dql_prompt_template = None

with open(ddl_prompt_path) as file:
    ddl_prompt_template = file.read()
with open(dml_dql_prompt_path) as file:
    dml_dql_prompt_template = file.read()

In [335]:
ddl_indicator_phrases = [
    'This class is generated by jOOQ',
    'extends TableImpl',
    'extends org.jooq.impl.TableImpl'
]

def prepare_prompt(file_path):
    with open(file_path) as file:
        lines = file.readlines()
    with open(find_closest_keys_file(file_path)) as file:
        keys_lines = file.readlines()

    numbered_contents = "\n".join(
        f"{idx + 1}: {line.strip()}" for idx, line in enumerate(lines)
    )
    keys_file_contents = "\n".join(line.strip() for line in keys_lines)

    is_ddl = any(phrase in numbered_contents for phrase in ddl_indicator_phrases)
    prompt_template = ddl_prompt_template if is_ddl else dml_dql_prompt_template
    prompt_template = prompt_template.replace('FILE_CONTENTS', numbered_contents)
    prompt_template = prompt_template.replace('KEYS_CONTENTS', keys_file_contents)
    return prompt_template

In [338]:
from openai import AsyncOpenAI
import asyncio
from aiolimiter import AsyncLimiter
from pydantic import BaseModel
from enum import Enum
import json
import time

api_limiter = AsyncLimiter(100, 60)
client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    timeout=60
)

class AntipatternName(str, Enum):
    id_required = 'ID Required',
    keyless_entry = 'Keyless Entry',
    rounding_errors = 'Rounding Errors',
    thirty_one_flavors = '31 Flavors',
    poor_mans_search_engine = 'Poor Man’s Search Engine',
    implicit_columns = 'Implicit Columns',
    beware_of_the_unknown = 'Beware of the Unknown'

class AntipatternOccurrence(BaseModel):
    antipatternName: AntipatternName
    linesRangeStart: int
    linesRangeEnd: int
    codeFragment: str
    reasoning: str

class Response(BaseModel):
    occurrences: list[AntipatternOccurrence]

progress_lock = asyncio.Lock()
processed_count = 0
total_files = len(applicable_file_paths)
global_input_tokens = 0
global_output_tokens = 0
global_total_tokens = 0
global_total_cost = 0.0

async def generate(file_path):
    global processed_count, global_input_tokens, global_output_tokens, global_total_tokens, global_total_cost

    try:
        async with api_limiter:
            prompt = prepare_prompt(file_path)
            response = await client.responses.parse(
                model=model,
                reasoning={"effort": "xhigh"},
                temperature=0.0,
                text_format=Response,
                input=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            async with progress_lock:
                if response.usage:
                    global_input_tokens += response.usage.input_tokens
                    global_output_tokens += response.usage.output_tokens
                    global_total_tokens += response.usage.total_tokens
                    global_total_cost += response.usage.cost

            if response.output_parsed == None:
                # retry if result is empty (issue with Kimi K2 and some other models)
                if debug:
                    print("Received empty response")
                    print(response)
                return await generate(file_path)
                
            async with progress_lock:
                processed_count += 1
                if debug:
                    print(f"[{processed_count}/{total_files}] File analyzed successfully: {file_path}")

            return {
                "file": file_path,
                "response": response.output_parsed,
                "usage": response.usage,
                "status": "success"
            }
    except Exception as e:
        if debug:
            print(e)
        return await generate(file_path)

start_time = time.perf_counter()
tasks = [generate(fp) for fp in applicable_file_paths]
results = await asyncio.gather(*tasks)
end_time = time.perf_counter()

total_runtime = end_time - start_time
success_count = sum(1 for res in results if res["status"] == "success")
error_count = total_files - success_count

print("\n--- Analysis Summary ---")
print(f"Total Runtime:      {total_runtime:.2f} seconds")
print(f"Files Processed:    {success_count} success, {error_count} failed")
print(f"Input Tokens:       {global_input_tokens:,}")
print(f"Output Tokens:      {global_output_tokens:,}")
print(f"Total Tokens:       {global_total_tokens:,}")
print(f"Total Cost:         ${global_total_cost:.6f}")

[1/15] File analyzed successfully: repositories/NevinTure_java-course-2024/scrapper-jooq/src/main/java/edu/java/jooq/JooqCodegen.java
[2/15] File analyzed successfully: repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/repositories/jooq/JooqStackOverFlowQuestionRepository.java
[3/15] File analyzed successfully: repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/configuration/JooqConfig.java
[4/15] File analyzed successfully: repositories/NevinTure_java-course-2024/scrapper/src/test/java/edu/java/scrapper/services/JooqStackOverFlowQuestionServiceTest.java
[5/15] File analyzed successfully: repositories/NevinTure_java-course-2024/scrapper/src/test/java/edu/java/scrapper/services/JooqGitRepositoryServiceTest.java
[6/15] File analyzed successfully: repositories/NevinTure_java-course-2024/scrapper/src/main/java/edu/java/scrapper/repositories/jooq/JooqGitRepoRepository.java
[7/15] File analyzed successfully: repositories/NevinTure_

In [ ]:
results_df = pd.DataFrame(columns = ["Antipattern", "File", "Line from", "Line to"])

for result in results:
    if result['status'] == "success":
        for occurrence in result['response'].occurrences:
            results_df.loc[len(results_df)] = {
                "Antipattern": occurrence.antipatternName.value,
                "File": result["file"][(len(project_dir) + 1):],
                "Line from": occurrence.linesRangeStart,
                "Line to": occurrence.linesRangeEnd
            }
    else:
        print(f"Error: {result['error']}")

results_df.sort_values(by=["Antipattern", "File", "Line from"], ascending=True, ignore_index=True)

In [ ]:
import pandas as pd
import numpy as np

# --- 1. The Fixed Helper Functions ---

def iou_1d_inclusive(gt, pred):
    """
    Calculates Intersection over Union for DISCRETE inclusive intervals (lines).
    Math: Length = end - start + 1
    """
    # gt and pred are tuples/lists: (start, end)
    
    # 1. Calculate Intersection
    start_inter = max(gt[0], pred[0])
    end_inter = min(gt[1], pred[1])
    
    # We add +1 here because lines are discrete. 
    # If start=79, end=79, overlap is 1 line.
    inter_len = max(0, end_inter - start_inter + 1)
    
    # 2. Calculate Union
    # Union = Area_A + Area_B - Intersection
    len_gt = gt[1] - gt[0] + 1
    len_pred = pred[1] - pred[0] + 1
    
    union_len = len_gt + len_pred - inter_len
    
    return inter_len / union_len if union_len > 0 else 0.0

def calculate_counts(gt_intervals, pred_intervals, threshold=0.5):
    """
    Returns (True Positives for Recall, True Positives for Precision)
    """
    # 1. Calculate Recall Matches (Did we find the GT?)
    tp_recall_count = 0
    for g in gt_intervals:
        # If this GT overlaps sufficiently with ANY prediction
        if any(iou_1d_inclusive(g, p) >= threshold for p in pred_intervals):
            tp_recall_count += 1
            
    # 2. Calculate Precision Matches (Was this Prediction correct?)
    tp_precision_count = 0
    for p in pred_intervals:
        # If this Prediction overlaps sufficiently with ANY GT
        if any(iou_1d_inclusive(g, p) >= threshold for g in gt_intervals):
            tp_precision_count += 1
            
    return tp_recall_count, tp_precision_count

# --- 2. Data Preparation ---

# Ensure strict types
project_ground_truth_df['Line from'] = project_ground_truth_df['Line from'].astype(int)
project_ground_truth_df['Line to'] = project_ground_truth_df['Line to'].astype(int)
results_df['Line from'] = results_df['Line from'].astype(int)
results_df['Line to'] = results_df['Line to'].astype(int)

# Strip whitespace from filenames just in case
project_ground_truth_df['File'] = project_ground_truth_df['File'].str.strip()
results_df['File'] = results_df['File'].str.strip()

# --- 3. Metric Calculation ---

all_antipatterns = set(project_ground_truth_df['Antipattern']).union(set(results_df['Antipattern']))
metric_rows = []

for ap in all_antipatterns:
    # Filter Dataframes
    gt_subset = project_ground_truth_df[project_ground_truth_df['Antipattern'] == ap]
    pred_subset = results_df[results_df['Antipattern'] == ap]
    
    # Get all relevant files for this antipattern
    relevant_files = set(gt_subset['File']).union(set(pred_subset['File']))
    
    ap_tp_recall = 0   
    ap_tp_precision = 0 
    ap_total_gt = 0    
    ap_total_pred = 0  
    
    for filename in relevant_files:
        # Extract intervals
        gts = gt_subset[gt_subset['File'] == filename][['Line from', 'Line to']].values.tolist()
        preds = pred_subset[pred_subset['File'] == filename][['Line from', 'Line to']].values.tolist()
        
        ap_total_gt += len(gts)
        ap_total_pred += len(preds)
        
        # Calculate Matches using the Inclusive IoU
        matches_rec, matches_prec = calculate_counts(gts, preds, threshold=0.5)
        
        ap_tp_recall += matches_rec
        ap_tp_precision += matches_prec

    # Calculate Rates
    precision = ap_tp_precision / ap_total_pred if ap_total_pred > 0 else 0.0
    recall = ap_tp_recall / ap_total_gt if ap_total_gt > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
    metric_rows.append({
        "Antipattern": ap,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Support (GT Count)": ap_total_gt,
        "Pred Count": ap_total_pred
    })

# --- 4. Weighted Average and Output ---

metrics_df = pd.DataFrame(metric_rows)
total_support = metrics_df["Support (GT Count)"].sum()

if total_support > 0:
    w_avg_prec = (metrics_df["Precision"] * metrics_df["Support (GT Count)"]).sum() / total_support
    w_avg_rec = (metrics_df["Recall"] * metrics_df["Support (GT Count)"]).sum() / total_support
    w_avg_f1 = (metrics_df["F1-Score"] * metrics_df["Support (GT Count)"]).sum() / total_support
else:
    w_avg_prec, w_avg_rec, w_avg_f1 = 0.0, 0.0, 0.0

metrics_df.loc[len(metrics_df)] = {
    "Antipattern": "WEIGHTED AVERAGE",
    "Precision": w_avg_prec,
    "Recall": w_avg_rec,
    "F1-Score": w_avg_f1,
    "Support (GT Count)": total_support,
    "Pred Count": metrics_df["Pred Count"].sum()
}

# Display
metrics_df.round(4)